<a href="https://colab.research.google.com/github/shreyaganesh-123/CSA6102-DIGITAL-FORENSICS/blob/main/lab_exoeriments_digital_forensics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

EXP 1

In [ ]:
import os
import shutil
import time

# Create an "original" piece of evidence
with open("evidence_original.txt", "w") as f:
    f.write("Case File #001 - Suspect device log")

orig_stat = os.stat("evidence_original.txt")

print("Original file - created (ctime):", time.ctime(orig_stat.st_ctime))
print("Original file - modified (mtime):", time.ctime(orig_stat.st_mtime))

# Simulate an investigator copying the file to a workstation
time.sleep(1)

shutil.copy2("evidence_original.txt", "evidence_copy.txt")

copy_stat = os.stat("evidence_copy.txt")

print("\nCopied file - created (ctime):", time.ctime(copy_stat.st_ctime))
print("Copied file - modified (mtime):", time.ctime(copy_stat.st_mtime))

print("\nLocard's Exchange Principle: the copy operation itself created a new")
print("ctime on the destination file - every interaction leaves a trace.")

Original file - created (ctime): Tue Jul 14 07:03:07 2026
Original file - modified (mtime): Tue Jul 14 07:03:07 2026

Copied file - created (ctime): Tue Jul 14 07:03:08 2026
Copied file - modified (mtime): Tue Jul 14 07:03:07 2026

Locard's Exchange Principle: the copy operation itself created a new
ctime on the destination file - every interaction leaves a trace.


EXP 2


In [ ]:
import hashlib
import datetime

def sha256_of(filename):
    with open(filename, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

# Step 1: Investigator seizes evidence and hashes it immediately
with open("evidence.txt", "w") as f:
    f.write("Suspect chat log: meeting at 10pm, bring the drive.")

seizure_hash = sha256_of("evidence.txt")

custody_log = []
custody_log.append(
    f"{datetime.datetime.now()} - SEIZED by Officer A - hash={seizure_hash}"
)

# Step 2: Evidence is later handed to a forensic analyst; verify integrity
handoff_hash = sha256_of("evidence.txt")

if handoff_hash == seizure_hash:
    custody_log.append(
        f"{datetime.datetime.now()} - RECEIVED by Analyst B - integrity VERIFIED"
    )
else:
    custody_log.append(
        f"{datetime.datetime.now()} - RECEIVED by Analyst B - integrity FAILED (tampered!)"
    )

print("=== Chain of Custody Log ===")

for entry in custody_log:
    print(entry)

=== Chain of Custody Log ===
2026-07-14 07:04:02.173465 - SEIZED by Officer A - hash=b7eb05cf41efbcc50adc36b12679ce4f12de58460e59500eb6bb62858810f84c
2026-07-14 07:04:02.173635 - RECEIVED by Analyst B - integrity VERIFIED


EXP 3

In [1]:
import re

def check_email(sender, subject, body):
    flags = []

    # Check for urgency/phishing keywords
    if re.search(r"(urgent|verify your account|suspended|click here)", body, re.IGNORECASE):
        flags.append("Urgency/pressure language detected")

    # Check for raw IP address links
    if re.search(r"\d{1,3}(?:\.\d{1,3}){3}", body):
        flags.append("Raw IP address link found in body")

    # Check suspicious domain patterns
    domain = sender.split("@")[-1]
    if any(k in domain.lower() for k in ["secure", "verify", "update"]) and "@gmail" not in sender:
        flags.append("Suspicious sender domain naming pattern")

    return flags


# Sample emails
emails = [
    ("support@paypal.com", "Your monthly statement", "Please find your statement attached."),
    ("alert@paypal-secure-verify.com", "URGENT: Verify your account",
     "Click here: http://192.168.10.5/login"),
]

# Process emails
for sender, subject, body in emails:
    issues = check_email(sender, subject, body)
    verdict = "PHISHING SUSPECTED" if issues else "Looks legitimate"

    print(f"\nFrom: {sender}\nSubject: {subject}\nVerdict: {verdict}")

    for i in issues:
        print(" -", i)


From: support@paypal.com
Subject: Your monthly statement
Verdict: Looks legitimate

From: alert@paypal-secure-verify.com
Subject: URGENT: Verify your account
Verdict: PHISHING SUSPECTED
 - Urgency/pressure language detected
 - Raw IP address link found in body
 - Suspicious sender domain naming pattern


EXP 4

In [2]:
import socket

domains = ["www.google.com", "www.python.org", "notarealdomain12345.com"]

print("OSINT Domain Reconnaissance")

for d in domains:
    try:
        ip = socket.gethostbyname(d)
        print(f"{d:30s} -> {ip}")
    except socket.gaierror:
        print(f"{d:30s} -> Could not resolve (invalid/unreachable)")

OSINT Domain Reconnaissance
www.google.com                 -> 142.251.151.119
www.python.org                 -> 151.101.0.223
notarealdomain12345.com        -> Could not resolve (invalid/unreachable)


EXP 5

In [3]:
import datetime

def investigation_report(case_id, evidence_list):
    stages = {
        "Identification": f"Incident reported for case {case_id}. Devices/logs identified for review.",
        "Collection": f"{len(evidence_list)} item(s) collected: {', '.join(evidence_list)}",
        "Preservation": "All items hashed (SHA-256) and stored in a write-protected evidence folder.",
        "Analysis": "Log files and file metadata examined for indicators of compromise.",
        "Reporting": "Findings compiled into a structured forensic report for legal review.",
    }

    print(f"=== Investigation Lifecycle: Case {case_id} ===")
    print(f"Generated: {datetime.datetime.now()}\n")

    for stage, detail in stages.items():
        print(f"[{stage}]")
        print(f"  {detail}\n")


# Function call
investigation_report(
    "CASE-2026-014",
    ["laptop_disk_image.dd", "router_traffic.pcap", "email_headers.txt"]
)

=== Investigation Lifecycle: Case CASE-2026-014 ===
Generated: 2026-07-25 03:49:11.491411

[Identification]
  Incident reported for case CASE-2026-014. Devices/logs identified for review.

[Collection]
  3 item(s) collected: laptop_disk_image.dd, router_traffic.pcap, email_headers.txt

[Preservation]
  All items hashed (SHA-256) and stored in a write-protected evidence folder.

[Analysis]
  Log files and file metadata examined for indicators of compromise.

[Reporting]
  Findings compiled into a structured forensic report for legal review.



EXP 6

In [4]:
import hashlib

def sha256_of(filename):
    with open(filename, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


# Create a sample "disk" file (simulating a small storage device)
with open("original_disk.img", "wb") as f:
    f.write(b"HEADER" + bytes(range(256)) * 4 + b"FOOTER")


# Step 1: Create a bit-for-bit forensic copy (bit-stream image)
with open("original_disk.img", "rb") as src, open("forensic_copy.img", "wb") as dst:
    dst.write(src.read())


# Step 2: Verify the copy is identical using hashing
original_hash = sha256_of("original_disk.img")
copy_hash = sha256_of("forensic_copy.img")

print("Original image hash:", original_hash)
print("Forensic copy hash :", copy_hash)

print(
    "Match:",
    "VERIFIED - exact bit-stream copy"
    if original_hash == copy_hash
    else "MISMATCH - copy corrupted"
)

Original image hash: d50630bee2bf9926344781042d10cb09377730818efda8e80f9eeec21008a9b0
Forensic copy hash : d50630bee2bf9926344781042d10cb09377730818efda8e80f9eeec21008a9b0
Match: VERIFIED - exact bit-stream copy


EXP 7

In [5]:
SIGNATURES = {
    b"\xFF\xD8\xFF": "JPEG image",
    b"\x89PNG": "PNG image",
    b"%PDF": "PDF document",
    b"PK\x03\x04": "ZIP archive (or .docx/.xlsx)",
}

def identify_file(path):
    with open(path, "rb") as f:
        header = f.read(8)

    for sig, filetype in SIGNATURES.items():
        if header.startswith(sig):
            return filetype

    return "Unknown file type"


# Create sample files with fake/renamed extensions to test signature detection

with open("photo.txt", "wb") as f:  # renamed JPEG
    f.write(b"\xFF\xD8\xFF\xE0" + b"\x00" * 20)

with open("document.dat", "wb") as f:  # renamed PDF
    f.write(b"%PDF-1.4" + b"\x00" * 20)


# Test detection
for filename in ["photo.txt", "document.dat"]:
    print(f"{filename:15s} -> Actual type: {identify_file(filename)}")

photo.txt       -> Actual type: JPEG image
document.dat    -> Actual type: PDF document
